# cMAB Simulation

This notebook shows a simulation framework for the contextual multi-armed bandit (cMAB). It allows to study the behaviour of the bandit algoritm, to evaluate results and to run experiments on simulated data under different context, reward and action settings.

In [1]:
from sklearn.datasets import make_classification

from pybandits.cmab import CmabBernoulli
from pybandits.cmab_simulator import CmabSimulator
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


First we need to define the simulation parameters. The parameters are split into two parts. The general parameters contain:
- Number of update rounds
- Number of samples per batch of update round
- Seed for reproducibility
- Verbosity enabler
- Visualization enabler

The problem definition parameters contain:
- Number of groups
- Number of features

Data are processed in batches of size n>=1. Per each batch of simulated samples, the cMAB selects one action and collects the corresponding simulated reward for each sample. Then, prior parameters are updated based on returned rewards from recommended actions.

In [2]:
# general simulator parameters
n_updates = 5
batch_size = 100
random_seed = None
verbose = True
visualize = True

In [3]:
# problem definition simulation parameters
n_groups = 3
n_features = 5

Next, we initialize the context matrix $X$ and the groups of samples. Samples that belong to the same group have features that come from the same distribution.
Then, the action model and the cMAB are defined. We define three actions, each with a Bayesian Logistic Regression model. The model is defined by a Student-T prior for the intercept and a Student-T prior for each feature coefficient.

In [4]:
# init context matrix and groups

context, group = make_classification(
    n_samples=batch_size * n_updates, n_features=n_features, n_informative=n_features, n_redundant=0, n_classes=n_groups
)
group = [str(g) for g in group]

In [5]:
# define action model


def create_bnn(n_features, bias_mu, bias_sigma, update_method, update_kwargs):
    """Create a BayesianNeuralNetwork with given parameters."""
    bias = StudentTArray.cold_start(mu=bias_mu, sigma=bias_sigma, shape=1)
    weight = StudentTArray.cold_start(shape=(n_features, 1))
    layer_params = BnnLayerParams(weight=weight, bias=bias)
    model_params = BnnParams(bnn_layer_params=[layer_params])
    feature_config = FeaturesConfig(n_features=n_features)
    return BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    )


update_method = "VI"
update_kwargs = {"num_steps": 10, "batch_size": 32, "optimizer_type": "adam"}
blr_kwargs = dict(
    n_features=n_features, bias_mu=1, bias_sigma=2, update_method=update_method, update_kwargs=update_kwargs
)
actions = {
    "a1": create_bnn(**blr_kwargs),
    "a2": create_bnn(**blr_kwargs),
    "a3": create_bnn(**blr_kwargs),
}
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

Finally, we need to define the probabilities of positive rewards per each action/group, i.e. the ground truth ('Action A': 0.8 for group '0' means that if the bandits selects 'Action A' for samples that belong to group '0', then the environment will return a positive reward with 80% probability).


In [6]:
# init probability of rewards randomly using splines
probs_reward = None

Now, we initialize the cMAB as shown in the previous notebook and the CmabSimulator with the parameters set above.

In [7]:
# init simulation
cmab_simulator = CmabSimulator(
    mab=cmab,
    group=group,
    batch_size=batch_size,
    n_updates=n_updates,
    probs_reward=probs_reward,
    context=context,
    verbose=verbose,
)

Now, we can start simulation process by executing run() which performs the following steps:
```
For i=0 to n_updates:
    Extract batch[i] of samples from X
    Model recommends the best actions as the action with the highest reward probability to each simulated sample in batch[i] and collect corresponding simulated rewards
    Model priors are updated using information from recommended actions and returned rewards
```
Finally, we can visualize the results of the simulation. As defined in the ground truth: 'a2' was the action recommended the most for samples that belong to group '0', 'a1' to group '1' and both 'a1' and 'a3' to group '2'.

In [8]:
cmab_simulator.run()

/home/runner/work/pybandits/pybandits/pybandits/simulator.py:229: RuntimeWarning: overflow encountered in exp
  return np.where(s >= 0, 1 / (1 + np.exp(-s)), np.exp(s) / (1 + np.exp(s))).item()
/home/runner/work/pybandits/pybandits/pybandits/simulator.py:229: RuntimeWarning: invalid value encountered in scalar divide
  return np.where(s >= 0, 1 / (1 + np.exp(-s)), np.exp(s) / (1 + np.exp(s))).item()
/home/runner/work/pybandits/pybandits/pybandits/simulator.py:229: RuntimeWarning: overflow encountered in exp
  return np.where(s >= 0, 1 / (1 + np.exp(-s)), np.exp(s) / (1 + np.exp(s))).item()
/home/runner/work/pybandits/pybandits/pybandits/simulator.py:229: RuntimeWarning: invalid value encountered in scalar divide
  return np.where(s >= 0, 1 / (1 + np.exp(-s)), np.exp(s) / (1 + np.exp(s))).item()
/home/runner/work/pybandits/pybandits/pybandits/simulator.py:335: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this wil

/home/runner/work/pybandits/pybandits/pybandits/model.py:1568: UserWarning: subsample_size does not match len(subsample), 32 vs 30. Did you accidentally use different subsample_size in the model and guide?
  with numpyro.plate("data", N, subsample_size=batch_size) as idx:


SVI:   0%|          | 0/10 [00:00<?, ?it/s]

SVI:  10%|█         | 1/10 [00:00<00:05,  1.73it/s]

SVI:  10%|█         | 1/10 [00:00<00:05,  1.73it/s, loss=127.7840]

...

SVI:  90%|█████████ | 9/10 [00:00<00:00,  1.34it/s, loss=124.1494]

SVI: 100%|██████████| 10/10 [00:00<00:00,  1.34it/s, loss=140.2147]

2026-04-23 11:02:25.390 | INFO     | pybandits.simulator:_print_results:541 - Simulation results (first 10 observations):



2026-04-23 11:02:25.413 | INFO     | pybandits.simulator:_print_results:542 - Count of actions selected by the bandit: 



2026-04-23 11:02:25.416 | INFO     | pybandits.simulator:_print_results:543 - Observed proportion of positive rewards for each action:



Furthermore, we can examine the number of times each action was selected and the proportion of positive rewards for each action.

In [9]:
cmab_simulator.selected_actions_count

,action,a1,a2,a3,cum_a1,cum_a2,cum_a3
group,batch,,,,,,
0,0.0,18,10,6,18,10,6
1,0.0,8,11,12,8,11,12
2,0.0,4,16,15,4,16,15
0,1.0,5,16,7,23,26,13
1,1.0,9,13,17,17,24,29
2,1.0,3,10,20,7,26,35
0,2.0,4,17,14,27,43,27
1,2.0,5,10,15,22,34,44
2,2.0,5,11,19,12,37,54


In [10]:
cmab_simulator.positive_reward_proportion

proportion
action group           
a1     0            0.5
       1       0.941176
       2       0.736842
a2     0       0.138889
       1            0.4
       2       0.050847
a3     0        0.40678
       1       0.217949
       2       0.011364